# Spark Environment & Execution

This notebook explains how a Spark application starts, how PySpark communicates with the Spark engine, and how Spark converts our code into jobs, stages, and tasks.


## Learning objectives

- What a Spark environment contains
- Spark local and cluster execution environments
- Spark application, driver, executor, core, partition, and task
- SparkSession and SparkContext
- PySpark and JVM communication
- Transformations, actions, and lazy evaluation
- Job, stage, and task creation
- Spark execution plans
- Spark UI navigation
- Client mode and cluster mode
- Common execution mistakes

# 1. What Is a Spark Environment?

A **Spark environment** is the complete setup required to write, submit, execute, monitor, and troubleshoot a Spark application.

```text
Spark Environment
│
├── Programming Language
│   ├── Python / PySpark
│   ├── Scala
│   ├── Java
│   └── R
│
├── Spark Runtime
│   ├── Spark Core
│   ├── Spark SQL
│   ├── Structured Streaming
│   ├── MLlib
│   └── GraphX
│
├── Execution Environment
│   ├── Local Mode
│   ├── Standalone Cluster
│   ├── YARN
│   ├── Kubernetes
│   └── Managed Cloud Platforms
│
├── Storage
│   ├── Local File System
│   ├── HDFS
│   ├── Amazon S3
│   ├── Azure Data Lake
│   └── Google Cloud Storage
│
└── Monitoring
    ├── Spark UI
    ├── Driver Logs
    ├── Executor Logs
    └── Cluster Monitoring
```

In simple terms:

> The Spark environment determines where the Spark application runs, where it reads and writes data, what resources it receives, and how it is monitored.

# 2. Spark Application Execution Overview

A PySpark application moves through the following logical flow:

```text
PySpark Code
    │
    ▼
Python Process
    │
    ▼
SparkSession
    │
    ▼
Spark Driver
    │
    ▼
Logical Plan
    │
    ▼
Optimized Logical Plan
    │
    ▼
Physical Plan
    │
    ▼
Job
    │
    ▼
Stages
    │
    ▼
Tasks
    │
    ▼
Executor Threads / CPU Cores
```

# 3. Import PySpark and Create a SparkSession

`SparkSession` is the primary entry point for modern Spark applications.

The `spark` object allows us to:

- Create DataFrames
- Read and write files
- Run Spark SQL
- Access configuration
- Access the SparkContext
- Manage tables and catalogs
- Cache and uncache data
- Stop the Spark application

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count

spark = (
    SparkSession.builder
    .appName("SparkEnvironmentAndExecution")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print("SparkSession created successfully.")

SparkSession created successfully.


In [6]:
sc=spark.sparkContext
print("Spark UI URL:", sc.uiWebUrl)

Spark UI URL: http://macbookair.lan:4040


## SparkSession builder 

```python
SparkSession.builder
```

Starts building a SparkSession.

```python
.appName("SparkEnvironmentAndExecution")
```

Sets the application name. This name appears in the Spark UI, logs, and cluster manager.

```python
.master("local[2]")
```

Runs Spark locally using two logical CPU threads.

```python
.config("spark.sql.shuffle.partitions", "4")
```

Sets the default number of partitions produced by many shuffle operations.

```python
.getOrCreate()
```

Returns an existing SparkSession if one already exists; otherwise, it creates a new one.

# 4. Inspect the Spark Environment

The SparkSession contains access to Spark configuration and the underlying SparkContext.

In [3]:
sc = spark.sparkContext

print("Spark Version:", spark.version)
print("Application Name:", sc.appName)
print("Application ID:", sc.applicationId)
print("Master:", sc.master)
print("Default Parallelism:", sc.defaultParallelism)
print("Spark UI URL:", sc.uiWebUrl)
print("Shuffle Partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Spark Version: 3.5.7
Application Name: SparkEnvironmentAndExecution
Application ID: local-1785770977656
Master: local[2]
Default Parallelism: 2
Spark UI URL: http://macbookair.lan:4040
Shuffle Partitions: 4


## Important observations

- `Spark Version` tells us which Spark runtime is active.
- `Application Name` identifies the application.
- `Application ID` uniquely identifies the running Spark application.
- `Master` tells us where Spark is running.
- `Default Parallelism` gives a default task parallelism value for low-level Spark operations.
- `Spark UI URL` shows the monitoring address.
- `spark.sql.shuffle.partitions` controls the default number of shuffle partitions for DataFrame and SQL operations.

# 5. Spark Application

A **Spark application** is the complete user program submitted to Spark.

```text
Spark Application
│
├── One Driver Process
│
└── Zero or More Executor Processes
```

In local mode, driver and executor activity run on the same computer.

In a distributed cluster, the driver and executors may run on separate machines.

# 6. Driver

The **driver** is the main controlling process of a Spark application.

The driver is responsible for:

- Starting the application
- Creating the SparkSession
- Converting code into execution plans
- Creating jobs, stages, and tasks
- Requesting resources
- Scheduling tasks
- Tracking executor progress
- Handling failures
- Collecting results
- Hosting the Spark UI

```text
User Program
     │
     ▼
Spark Driver
     │
     ├── Understands the code
     ├── Creates execution plans
     ├── Divides work
     ├── Assigns tasks
     ├── Tracks execution
     └── Returns results
```

The driver is often described as the **brain of the Spark application**.

# 7. Executor

An **executor** is a process responsible for performing the actual data processing.

Executors:

- Execute tasks
- Read partitions
- Apply transformations
- Perform joins and aggregations
- Cache data
- Write output
- Return results and status to the driver

```text
Driver
  │
  ├── Task 1 ──► Executor 1
  ├── Task 2 ──► Executor 1
  ├── Task 3 ──► Executor 2
  └── Task 4 ──► Executor 2
```

## Driver vs Executor

| Driver | Executor |
|---|---|
| Controls the application | Processes the data |
| Creates execution plans | Executes tasks |
| Creates jobs and stages | Reads and transforms partitions |
| Schedules tasks | Stores cached data |
| Usually one per application | Usually multiple per application |
| Driver failure can terminate the application | Failed tasks can usually be retried |

Analogy:

```text
Driver    = Project Manager
Executor  = Team Member
Task      = Assigned Work
Partition = Portion of Data
```

# 8. SparkContext

`SparkContext` is the lower-level entry point for Spark Core.

It manages:

- Connection to the execution environment
- RDD creation
- Task scheduling
- Cluster communication
- Broadcast variables
- Accumulators

Modern applications normally begin with SparkSession and access SparkContext through it.

In [ ]:
sc = spark.sparkContext

print("SparkContext Type:", type(sc))
print("Application Name:", sc.appName)
print("Master:", sc.master)

## SparkSession and SparkContext relationship

```text
SparkSession
│
├── DataFrame API
├── Spark SQL
├── Catalog
├── Table Operations
├── Configuration
│
└── SparkContext
    ├── RDD API
    ├── Task Scheduling
    ├── Cluster Communication
    ├── Broadcast Variables
    └── Accumulators
```

Use `spark` for DataFrame and SQL operations.

Use `sc` for lower-level RDD operations and Spark Core features.

# 9. Create a Sample DataFrame

In [ ]:
employee_data = [
    (1, "Aryan", "Engineering", 90000),
    (2, "Ravi", "Engineering", 75000),
    (3, "Priya", "HR", 65000),
    (4, "Neha", "Finance", 80000),
    (5, "Amit", "Engineering", 70000),
    (6, "Kiran", "Finance", 85000),
    (7, "Pooja", "HR", 72000),
    (8, "Rahul", "Engineering", 95000)
]

columns = [
    "employee_id",
    "employee_name",
    "department",
    "salary"
]

employee_df = spark.createDataFrame(employee_data, columns)

employee_df.show(truncate=False)

# 10. Inspect DataFrame Schema and Partitions

In [ ]:
employee_df.printSchema()

print("Number of input partitions:", employee_df.rdd.getNumPartitions())

## Partition

A **partition** is a logical chunk of distributed data.

Spark processes data partition by partition.

For one stage, Spark normally creates one task for each partition that must be processed.

```text
Partition 0 → Task 0
Partition 1 → Task 1
Partition 2 → Task 2
Partition 3 → Task 3
```

# 11. Transformation and Action

Spark operations are mainly classified into:

- **Transformations**
- **Actions**

A transformation creates a new DataFrame or RDD but does not normally execute the complete computation immediately.

Examples:

```python
df.filter(...)
df.select(...)
df.groupBy(...)
df.join(...)
```

An action requests a result or writes output and therefore triggers Spark execution.

Examples:

```python
df.show()
df.count()
df.collect()
df.write.parquet(...)
```

# 12. Lazy Evaluation 

Spark uses **lazy evaluation**.

Transformations build an execution plan. The plan is executed only when an action is called.

In [ ]:
print("Step 1: Before transformation")

filtered_df = employee_df.filter(col("salary") > 70000)

print("Step 2: Filter transformation created")

selected_df = filtered_df.select(
    "employee_id",
    "employee_name",
    "department",
    "salary"
)

print("Step 3: Select transformation created")
print("No complete Spark job has been requested by these transformations alone.")

The following `show()` call is an action and triggers Spark execution.

In [ ]:
selected_df.show(truncate=False)

Execution flow:

```text
filter()
   │
   ▼
select()
   │
   ▼
show()
   │
   ▼
Spark Job Starts
```

Lazy evaluation allows Spark to optimize the entire chain before processing the data.

# 13. Create a Transformation Pipeline

This pipeline:

1. Filters employees with salary greater than or equal to 75,000
2. Groups them by department
3. Counts employees
4. Calculates average salary
5. Orders the final result

In [ ]:
high_salary_df = employee_df.filter(col("salary") >= 75000)

department_summary_df = (
    high_salary_df
    .groupBy("department")
    .agg(
        count("*").alias("employee_count"),
        avg("salary").alias("average_salary")
    )
    .orderBy("department")
)

At this point, Spark has created a plan. The final result has not yet been requested.

# 14. Inspect the Execution Plan

For DataFrame and SQL operations, Spark creates multiple plans:

```text
User Code
   │
   ▼
Unresolved Logical Plan
   │
   ▼
Analyzed Logical Plan
   │
   ▼
Optimized Logical Plan
   │
   ▼
Physical Plan
   │
   ▼
RDD Execution
   │
   ▼
Tasks
```

Important concepts:

- **Analyzed logical plan:** validates tables, columns, functions, and data types.
- **Optimized logical plan:** applies optimizations such as predicate pushdown and column pruning.
- **Physical plan:** chooses actual operators such as hash aggregate, sort, exchange, and joins.

In [ ]:
department_summary_df.explain(mode="formatted")

Look for operators such as:

- `Scan ExistingRDD`
- `Filter`
- `HashAggregate`
- `Exchange`
- `Sort`
- `AdaptiveSparkPlan`

`Exchange` usually represents data redistribution and commonly indicates a shuffle.

# 15. Trigger the Action

In [ ]:
department_summary_df.show(truncate=False)

Conceptual execution:

```text
Step 1: Read employee partitions
           │
Step 2: Apply salary filter
           │
Step 3: Prepare department values
           │
Step 4: Shuffle records by department
           │
Step 5: Count and average by department
           │
Step 6: Sort the result
           │
Step 7: Return rows for show()
```

# 16. Job, Stage, and Task

When an action is executed, Spark creates one or more jobs.

```text
Application
│
├── Job 1
│   ├── Stage 1
│   │   ├── Task 1
│   │   ├── Task 2
│   │   └── Task 3
│   │
│   └── Stage 2
│       ├── Task 1
│       ├── Task 2
│       └── Task 3
│
└── Job 2
```

Definitions:

- **Application:** complete submitted Spark program
- **Job:** normally created by an action
- **Stage:** group of tasks that can run without crossing a shuffle boundary
- **Task:** smallest unit of execution
- **Partition:** portion of data processed by a task

# 17. CPU Core and Task Relationship

A CPU core generally runs one Spark task at a time.

```text
Executor
│
├── Core 1 → Task A
├── Core 2 → Task B
├── Core 3 → Task C
└── Core 4 → Task D
```

Therefore:

```text
Approximate Concurrent Tasks
=
Total Available Executor Cores
```

Example:

```text
5 Executors × 4 Cores = 20 Concurrent Tasks
```

If a stage has 100 tasks:

```text
100 Tasks ÷ 20 Concurrent Tasks = Approximately 5 Waves
```

This is a simplified estimate. Real execution also depends on skew, I/O, failures, scheduling delay, and resource availability.

# 18. Inspect Partition-to-Task Behavior

Create an RDD with four partitions and inspect which values belong to each partition.

In [ ]:
numbers_rdd = sc.parallelize(range(1, 21), 4)

print("Number of partitions:", numbers_rdd.getNumPartitions())

partition_contents = numbers_rdd.glom().collect()

for index, values in enumerate(partition_contents):
    print(f"Partition {index}: {values}")

`glom()` groups the elements of each partition into a local list. It is useful for learning, but should not be used carelessly on very large datasets.

# 19. Narrow and Wide Transformations

A **narrow transformation** does not normally require data to move across partitions.

Examples:

- `filter`
- `map`
- `select`
- `withColumn`

A **wide transformation** normally requires data redistribution across partitions.

Examples:

- `groupBy`
- `reduceByKey`
- `join`
- `distinct`
- `orderBy`

Wide transformations often introduce a shuffle and a new stage.

# 20. Spark Master URLs

The master URL defines where Spark executes.

| Master value | Meaning |
|---|---|
| `local` | One local thread |
| `local[1]` | One local thread |
| `local[2]` | Two local threads |
| `local[*]` | All available logical CPU cores |
| `spark://host:port` | Spark Standalone cluster |
| `yarn` | Hadoop YARN |
| `k8s://...` | Kubernetes |

For teaching, `local[2]` is useful because task waves are easier to observe.

# 21. Local Mode

In local mode, Spark runs on a single computer.

```text
Laptop / Desktop
│
├── Python Process
├── Driver JVM
├── Spark Scheduler
├── Executor Threads
├── CPU Cores
└── Local Memory
```

Local mode is useful for:

- Learning
- Development
- Debugging
- Unit testing
- Small datasets
- Notebook experiments

It is not appropriate for large production workloads requiring multiple machines.

# 22. Spark Standalone Cluster

Spark Standalone is Spark's built-in cluster manager.

```text
Spark Standalone Cluster
│
├── Master
│   └── Manages cluster resources
│
├── Worker 1
│   └── Executor processes
│
├── Worker 2
│   └── Executor processes
│
└── Worker 3
    └── Executor processes
```

Do not confuse the Standalone **Master** with the Spark **Driver**.

| Standalone Master | Spark Driver |
|---|---|
| Manages cluster resources | Manages one Spark application |
| Knows worker nodes | Knows jobs, stages, and tasks |
| Allocates cluster resources | Schedules application work |
| Belongs to the cluster | Belongs to one application |

# 23. YARN Execution Environment

YARN is Hadoop's cluster resource manager.

```text
YARN Cluster
│
├── ResourceManager
├── NodeManager
├── ApplicationMaster
└── Containers
```

When Spark runs on YARN, YARN allocates containers for the driver and executors based on deployment mode and configuration.

# 24. Kubernetes Execution Environment

Spark can run on Kubernetes.

```text
Kubernetes Cluster
│
├── Spark Driver Pod
├── Executor Pod 1
├── Executor Pod 2
├── Executor Pod 3
└── Kubernetes Scheduler
```

Kubernetes manages:

- Pod scheduling
- CPU and memory requests
- Container images
- Networking
- Pod restarts
- Resource isolation

# 25. Managed Spark Environments

Common managed Spark environments include:

- Amazon EMR
- AWS Glue
- Databricks
- Google Dataproc
- Azure Synapse
- Azure HDInsight

These platforms manage different parts of infrastructure provisioning, scaling, integration, and monitoring.

# 26. How PySpark Communicates with Spark

PySpark lets us write Spark code using Python, while much of the Spark engine runs in the JVM.

```text
Python Code
    │
    ▼
PySpark API
    │
    ▼
Communication Layer
    │
    ▼
Spark JVM
    │
    ▼
Spark Execution Engine
    │
    ▼
Executors
```

For built-in DataFrame operations, Python describes the required transformation and Spark executes the optimized plan through the Spark engine.

Example:

```python
df.filter(col("salary") > 50000)
```

Python is not normally processing each row directly.

## Python UDF consideration

A Python UDF may require communication and serialization between JVM execution and Python worker processes.

This can add overhead.

Prefer built-in Spark SQL functions whenever possible.

# 27. `spark-submit`

`spark-submit` is the standard command used to submit Spark applications.

Basic command:

```bash
spark-submit application.py
```

Local example:

```bash
spark-submit \
  --master "local[2]" \
  --driver-memory 2g \
  --conf spark.sql.shuffle.partitions=4 \
  application.py
```

Common options:

| Option | Meaning |
|---|---|
| `--master` | Execution environment |
| `--deploy-mode` | Driver location |
| `--driver-memory` | Driver memory |
| `--executor-memory` | Executor memory |
| `--executor-cores` | Cores per executor |
| `--num-executors` | Initial executor count in supported environments |
| `--conf` | Spark configuration |

# 28. What Happens During `spark-submit`?

```text
1. spark-submit starts
        │
2. Spark configuration is loaded
        │
3. Python process starts
        │
4. Driver JVM starts
        │
5. SparkSession is created
        │
6. Driver connects to the cluster manager
        │
7. Resources are requested
        │
8. Executors are started
        │
9. Application code is evaluated
        │
10. Actions create jobs
        │
11. Jobs are divided into stages
        │
12. Stages are divided into tasks
        │
13. Tasks process partitions
        │
14. Results are returned or written
        │
15. Spark application stops
```

# 29. Client Mode and Cluster Mode

The main difference is the location of the driver.

## Client mode

```text
Submission Machine
│
├── spark-submit
├── Driver
└── Spark UI
      │
      ▼
Cluster
├── Executor 1
├── Executor 2
└── Executor 3
```

Useful for interactive development and debugging.

## Cluster mode

```text
Submission Machine
│
└── Submits Application
        │
        ▼
Cluster
│
├── Driver
├── Executor 1
├── Executor 2
└── Executor 3
```

Commonly preferred for scheduled production jobs.

## Client mode vs cluster mode

| Client Mode | Cluster Mode |
|---|---|
| Driver runs on submission machine | Driver runs inside the cluster |
| Good for development | Good for production |
| Direct access to driver logs | Logs are managed by cluster platform |
| Submission machine must remain available | Submission machine can disconnect |
| Common for shells and notebooks | Common for scheduled batch jobs |

# 30. Spark UI

The Spark UI is commonly available at:

```text
http://localhost:4040
```

If that port is already used, Spark may use:

```text
http://localhost:4041
http://localhost:4042
```

In [ ]:
print("Open this Spark UI URL in your browser:")
print(sc.uiWebUrl)

## Spark UI tabs

### Jobs

Shows jobs, actions, duration, and associated stages.

### Stages

Shows tasks, input, output, shuffle read, shuffle write, failures, and timing.

### Storage

Shows cached DataFrames and RDDs.

### Environment

Shows Spark, JVM, and system properties.

### Executors

Shows task counts, memory, input, shuffle, and garbage collection metrics.

### SQL / DataFrame

Shows query plans and operator-level execution metrics.

# 31. Spark UI Exercise

Keep the SparkSession running and open the Spark UI.

Run the following action several times and inspect the Jobs and SQL tabs.

In [ ]:
department_summary_df.show(truncate=False)

Observe:

1. How many jobs were created?
2. Which action triggered the job?
3. How many stages were created?
4. Which stage has shuffle write?
5. Which stage has shuffle read?
6. How many tasks were created?
7. Can you find `Exchange` in the SQL plan?
8. Can you find `HashAggregate`?

# 32. Caching Demonstration

Caching stores reusable data so Spark does not need to recompute the entire lineage every time.

In [ ]:
cached_df = employee_df.filter(col("salary") >= 70000).cache()

print("First action:")
print("Count:", cached_df.count())

print("Second action:")
cached_df.show(truncate=False)

Inspect the **Storage** tab in the Spark UI.

The first action computes and stores the data. A later action may reuse the cached partitions.

In [ ]:
cached_df.unpersist()
print("Cached DataFrame removed from storage.")

# 33. Common Execution Mistakes

## Using `collect()` on large data

```python
records = df.collect()
```

This brings all records to the driver and can cause driver out-of-memory errors.

## Too few partitions

If there are 20 available cores but only 4 partitions, many cores may remain idle.

## Too many tiny partitions

This causes excessive task scheduling, metadata, and file overhead.

## Hardcoding the master

Avoid hardcoding `.master("local[*]")` in portable production code.

## Stopping Spark too early

After `spark.stop()`, a new SparkSession must be created before running more Spark operations.

## Ignoring the Spark UI

The Spark UI helps identify:

- Skew
- Slow stages
- Large shuffle
- Disk spill
- Failed tasks
- Garbage collection
- Underutilized executors

# 34. Why Avoid Hardcoding the Master in Production?

Learning code often uses:

```python
.master("local[*]")
```

Production-style code should usually use:

```python
spark = (
    SparkSession.builder
    .appName("ProductionApplication")
    .getOrCreate()
)
```

Then define the environment during submission:

```bash
spark-submit \
  --master yarn \
  --deploy-mode cluster \
  application.py
```

The same code can then move between development, testing, and production environments.

# 35. Real-Time Production Example

Assume a retail company receives 500 GB of daily sales data in Amazon S3.

```text
Amazon S3
   │
   ▼
Spark Application on EMR
   │
   ├── Driver
   │   ├── Reads configuration
   │   ├── Creates execution plan
   │   └── Schedules tasks
   │
   ├── Executor 1
   ├── Executor 2
   ├── Executor 3
   └── Executor 4
           │
           ▼
Cleaned and Aggregated Data
           │
           ▼
Amazon S3 / Redshift
```

Application flow:

1. Read sales data from S3
2. Remove invalid records
3. Join sales with product master data
4. Calculate revenue by region
5. Calculate daily product sales
6. Write Parquet output to S3
7. Load summarized data into Redshift
8. Send success or failure notification

# 36. Interview Questions and Answers

## What is a Spark application?

A Spark application is a complete user program submitted to Spark. It contains one driver and one or more executors depending on the execution environment.

## What is the role of the driver?

The driver runs the main application logic, creates execution plans, divides work into jobs, stages, and tasks, schedules tasks, and tracks executor progress.

## What is the role of an executor?

An executor processes partitions by executing tasks. It can also cache data and report results and status to the driver.

## What is SparkSession?

SparkSession is the main entry point for DataFrame, SQL, catalog, and modern Spark operations.

## What is SparkContext?

SparkContext is the lower-level component that connects the application to Spark Core and supports RDDs, scheduling, broadcast variables, and accumulators.

## What is lazy evaluation?

Spark delays transformation execution until an action requests a result. This allows Spark to optimize the complete plan.

## What happens when an action is called?

Spark analyzes the lineage, creates an optimized execution plan, creates jobs, divides jobs into stages and tasks, and sends tasks for execution.

## What is the relationship between partitions and tasks?

For a stage, Spark normally creates one task for every partition that needs to be processed.

## How many tasks can run simultaneously?

Approximately one task per available executor core.

## What is the difference between client and cluster mode?

In client mode, the driver runs on the submission machine. In cluster mode, the driver runs inside the cluster.

# 37. Knowledge Check

Answer these questions:

1. What starts a Spark job?
2. What is the difference between a transformation and an action?
3. Why does `groupBy()` normally create a new stage?
4. What does one Spark task usually process?
5. What limits the number of concurrent tasks?
6. Where does the driver run in client mode?
7. Where does the driver run in cluster mode?
8. Why can `collect()` be dangerous?
9. What does `Exchange` indicate in a physical plan?
10. Which Spark UI tab shows shuffle read and shuffle write?

# 38. Key Points to Remember

```text
1. SparkSession is the main entry point.
2. SparkContext provides lower-level Spark Core access.
3. Every Spark application has one driver.
4. Executors perform distributed processing.
5. Transformations are lazily evaluated.
6. Actions trigger Spark jobs.
7. Jobs are divided into stages.
8. Stages are divided into tasks.
9. One task normally processes one partition.
10. One CPU core generally runs one task at a time.
11. Shuffle boundaries normally create new stages.
12. Spark UI shows actual execution behavior.
13. Client and cluster mode mainly differ by driver location.
14. Local mode is suitable for learning and development.
15. Production code should avoid hardcoding the execution environment.
```

# 39. Optional Cleanup

Run this cell only after completing Spark UI observations.

In [ ]:
# Uncomment when you are completely finished with this notebook.
# spark.stop()

# Practical Session

In [ ]:
employees=spark.read.csv("employee.csv")
result=(employees.filter(col("salary") >= 50000)
        .groupby("department")
        .count())

In [7]:
spark.stop()